<a href="https://colab.research.google.com/github/franciscogarate/mcaf/blob/master/notebooks/Ejercicio_15_SCR_Vida.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/franciscogarate/mcaf

In [ ]:
import pandas as pd
from mcaf import clr
from itertools import islice

In [ ]:
!pip install pyliferisk

In [ ]:
from pyliferisk import MortalityTable, lx
from pyliferisk.mortalitytables import PASEM2020_Decesos_M_2ord

In [ ]:
!pip install xlsxwriter

In [ ]:
mt = MortalityTable(qx=PASEM2020_Decesos_M_2ord)
edad = 50

In [ ]:
def incr_capital(capital, t):
    return capital * (1 + 0.015) ** t

In [ ]:
LANZAMIENTOS = 6

In [ ]:
ESCENARIOS = {
    "base": {
        "FACTOR_QX": 0.,
        "FACTOR_CAT": 0.,
        "FACTOR_GGI": 0.,
        "FACTOR_LAP": 0.,
        "FACTOR_MAS": 0.
    },
    "mort": {
        "FACTOR_QX": 0.15,
        "FACTOR_CAT": 0.,
        "FACTOR_GGI": 0.,
        "FACTOR_LAP": 0.,
        "FACTOR_MAS": 0.
    },
    "cat": {
        "FACTOR_QX": 0.,
        "FACTOR_CAT": 1.5,
        "FACTOR_GGI": 0.,
        "FACTOR_LAP": 0.,
        "FACTOR_MAS": 0.
    },
    "gastos": {
        "FACTOR_QX": 0.,
        "FACTOR_CAT": 0.,
        "FACTOR_GGI": 0.11,
        "FACTOR_LAP": 0.,
        "FACTOR_MAS": 0.
    },
    "caida": {
        "FACTOR_QX": 0.,
        "FACTOR_CAT": 0.,
        "FACTOR_GGI": 0.,
        "FACTOR_LAP": 0.5,
        "FACTOR_MAS": 0.
    },
    "caida_masiva": {
        "FACTOR_QX": 0.,
        "FACTOR_CAT": 0.,
        "FACTOR_GGI": 0.,
        "FACTOR_LAP": 0,
        "FACTOR_MAS": 0.4
    },
}

In [ ]:
resumen = []
with pd.ExcelWriter(f'Ejercicio_15_SCR_Vida.xlsx', engine="xlsxwriter") as writer:
    for nombre_esc, params in islice(ESCENARIOS.items(), LANZAMIENTOS):
        FACTOR_QX = params['FACTOR_QX']
        FACTOR_CAT = params['FACTOR_CAT']
        FACTOR_GGI = params['FACTOR_GGI']
        FACTOR_LAP = params['FACTOR_LAP']
        FACTOR_MAS = params['FACTOR_MAS']
        # Proyecciones:
        df = pd.DataFrame(pd.date_range(start='2025-12-31', periods=(mt.w - edad), freq='YE'), columns=['Fecha'])
        df['edad'] = edad + df.index
        df['t'] = df.index
        df['lx'] = df['edad'].apply(lambda x: mt.lx[x + 1] if x <= mt.w else 0)
        df['qx'] = df['lx'].diff(-1).fillna(0) / df['lx'][0]
        df['sum_qx'] = df['qx'].cumsum()
        df['sum_qx'] = df['sum_qx'].apply(lambda x: min(x * (1 + FACTOR_QX), 1))
        if nombre_esc == "cat":
            df['sum_qx'] = df.apply(lambda x: x.sum_qx * FACTOR_CAT if x.t==1 else x.sum_qx, axis=1)
        df['px'] = 1 - df['sum_qx']
        df['capital'] = incr_capital(5000, df.t)
        df['pagos'] = df['capital'] * df['qx']
        df['caida'] = 0.03 * (1 + FACTOR_LAP)
        if nombre_esc == "caida_masiva":
            df['caida'] = df['t'].apply(lambda t: 0.03 if t!=0 else FACTOR_MAS)
        df['polizas'] = (1 - df['caida']).cumprod()
        df['clr'] = df['t'].apply(lambda t: clr[t])
        df['factor_desc'] = df.apply(lambda x: 1 / (1 + x.clr) ** (x.t), axis=1)
        df['polizas_benef'] = df['px'] * df['polizas']
        df['pagos_prob'] = df['pagos'] * df['polizas_benef']
        df['gastos'] = df['capital'] * 0.01 * (1 + FACTOR_GGI)
        df['gastos_prob'] = df['gastos'] * df['polizas_benef']
        df['salidas_prob'] = df['pagos_prob'] + df['gastos_prob']

        resultado = df.salidas_prob @ df.factor_desc
        print(resultado)
        resumen.append({"escenario": nombre_esc, "resultado": resultado})

        #total = pd.concat(output, ignore_index=True)
        df.to_excel(writer, sheet_name=nombre_esc, index=False)

    pd.DataFrame(resumen).to_excel(writer, sheet_name="resumen", index=False)